## Reading the Data

In [ ]:
# Import library yang diperlukan
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
import plotly.express as px
import missingno as msno
import random

In [ ]:
# Mengambil data dari google drive
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
# Membaca file user
user = pd.read_csv('/content/drive/MyDrive/Tim 28D Final Project Drive/updated_user.csv')

In [ ]:
user.head()

In [ ]:
# Melihat berapa banyak data
user.shape

In [ ]:
# Mengubah tipe data fitur-fitur yang diperlukan
user['join_date'] = pd.to_datetime(user['join_date'])
user['birth'] = pd.to_datetime(user['birth'])

In [ ]:
# Membaca tabel transaksi tiap provinsi
tr = pd.read_csv('/content/drive/MyDrive/Tim 28D Final Project Drive/updated_trx_merged.csv')

In [ ]:
# Hitung total gross amount dan discounts untuk setiap pengguna
tr_new = tr.groupby('user_id')[['gross_amount', 'discounts']].sum().reset_index()

In [ ]:
tr_new.head()

In [ ]:
# Melihat berapa banyak data
tr_new.shape

## Menggabungkan Data

In [ ]:
# Menggabungkan tabel user dan tabel transaksi
user_new = pd.merge(user, tr_new, left_on='id', right_on='user_id', how='inner')

In [ ]:
user_new.shape

In [ ]:
user_new.isnull().sum()

In [ ]:
# Tabel yang memiliki money spent dan gross amount 0 dan NaN (untuk dihapus/didrop)
user_noms_noga = user_new[(user_new['money_spent'] == 0) & (user_new['gross_amount'] == 0)]

In [ ]:
# Menghapus data yang tidak memiliki money spent dan gross amount
user_new = user_new.drop(user_noms_noga.index)

## Penghandle-an Data yang Lahir Lebih Dari 2006


### SC93

In [ ]:
user_SC93 = user_new[user_new['id'].str.contains('SC93')]

In [ ]:
user_SC93.shape

In [ ]:
# Check for negative values in 'money_spent'
minus_ms = user_SC93[user_SC93['money_spent'] < 0]

# Check for negative values in 'refund'
minus_refund = user_SC93[user_SC93['refund'] < 0]

# Check for negative values in 'wallet_balance'
minus_wb = user_SC93[user_SC93['wallet_balance'] < 0]

# Print the number of rows with negative values
print(f"Number of rows with negative money_spent: {len(minus_ms)}")
print(f"Number of rows with negative refund: {len(minus_refund)}")
print(f"Number of rows with negative wallet_balance: {len(minus_wb)}")

In [ ]:
# Drop rows with negative wallet balance
user_SC93 = user_SC93.drop(user_SC93[user_SC93['wallet_balance'] < 0].index)

In [ ]:
user93_over_birth = user_SC93[user_SC93['birth'] > '2006-12-31']

In [ ]:
user93_over_birth.shape

In [ ]:
# Membuat kolom baru 'year' yang berisi tahun dari kolom 'birth'
user_SC93['year'] = user_SC93['birth'].dt.year # Kolom year jika tidak diperlukan nantinya dapat dihapus

# Menghitung value counts untuk setiap tahun
value_counts_per_year = user_SC93['year'].value_counts().sort_index()

print(value_counts_per_year)

In [ ]:
# Membagi 2 bagian pada pada yang lahir lebih dari tahun 2006
# Setengahnya diisi dengan nilai mean dan setengah lagi diisi dengan nilai median
# Menghitung mean dan median dari DataFrame copy_user93
mean_value = user_SC93['birth'].mean()
median_value = user_SC93['birth'].median()

# Membagi user93_over_birth menjadi dua bagian
half_len = len(user93_over_birth) // 2
first_half = user93_over_birth.iloc[:half_len]
second_half = user93_over_birth.iloc[half_len:]

# Mengisi setengah pertama dengan mean dan setengah kedua dengan median
first_half['birth'] = mean_value
second_half['birth'] = median_value

# Menggabungkan kembali kedua bagian
user93_over_birth_filled = pd.concat([first_half, second_half])

# Hasilnya adalah DataFrame user93_over_birth yang telah diisi setengahnya dengan mean dan setengahnya dengan median
print(user93_over_birth_filled)

In [ ]:
user93_over_birth_filled['birth'] = user93_over_birth_filled['birth'].dt.date
user93_over_birth_filled.head()

In [ ]:
user93_over_birth_filled.dtypes

In [ ]:
user93_over_birth_filled['birth'] = pd.to_datetime(user93_over_birth_filled['birth'])

In [ ]:
# Memasukkan nilai yang sudah di-handle ke dalam tabel copy_user93
# Mengambil indeks data yang sama di kedua DataFrames
indexes = user93_over_birth_filled.index.intersection(user_SC93.index)

# Mengganti nilai pada copy_user93 dengan nilai yang sudah diperbaiki dari user93_over_birth_filled
user_SC93.loc[indexes, 'birth'] = user93_over_birth_filled.loc[indexes, 'birth']

In [ ]:
user_SC93 = user_SC93.drop('year', axis=1)

### SC33

In [ ]:
user_SC33 = user_new[user_new['id'].str.contains('SC33')]

In [ ]:
# Check for negative values in 'money_spent'
minus_ms = user_SC33[user_SC33['money_spent'] < 0]

# Check for negative values in 'refund'
minus_refund = user_SC33[user_SC33['refund'] < 0]

# Check for negative values in 'wallet_balance'
minus_wb = user_SC33[user_SC33['wallet_balance'] < 0]

# Check for min values in refund and wallet balance
minus_randwb = user_SC33.loc[(user_SC33['refund'] < 0) & (user_SC33['wallet_balance'] < 0)]

# Print the number of rows with negative values
print(f"Number of rows with negative money_spent: {len(minus_ms)}")
print(f"Number of rows with negative refund: {len(minus_refund)}")
print(f"Number of rows with negative wallet_balance: {len(minus_wb)}")
print(f"Number of rows with min refund and wallet balance: {len(minus_randwb)}")

In [ ]:
# Drop rows with negative refund
user_SC33 = user_SC33.drop(user_SC33[user_SC33['refund'] < 0].index)

# Drop rows with negative wallet balance
user_SC33 = user_SC33.drop(user_SC33[user_SC33['wallet_balance'] < 0].index)

In [ ]:
user33_over_birth = user_SC33[user_SC33['birth'] > '2006-12-31']

In [ ]:
# Membuat kolom baru 'year' yang berisi tahun dari kolom 'birth'
user_SC33['year'] = user_SC33['birth'].dt.year # Kolom year jika tidak diperlukan nantinya dapat dihapus

# Menghitung value counts untuk setiap tahun
value_counts_per_year = user_SC33['year'].value_counts().sort_index()

print(value_counts_per_year)

In [ ]:
# Membagi data pada provinsi SC33 yang lahir lebih dari 2006 menjadi 5 bagian
first_part = user33_over_birth.iloc[:2101]
second_part = user33_over_birth.iloc[2101:4202]
third_part = user33_over_birth.iloc[4202:6303]
fourth_part = user33_over_birth.iloc[6303:8404]
fifth_part = user33_over_birth.iloc[8404:]

first_part['birth'] = '2001-03-03'
second_part['birth'] = '2002-04-04'
third_part['birth'] = '2003-05-05'
fourth_part['birth'] = '2004-06-06'
fifth_part['birth'] = '2005-07-07'

user33_over_birth_filled = pd.concat([first_part, second_part, third_part, fourth_part, fifth_part])

# Hasilnya adalah DataFrame user93_over_birth yang telah diisi setengahnya dengan mean dan setengahnya dengan median
print(user33_over_birth_filled)

In [ ]:
user33_over_birth_filled['birth'] = pd.to_datetime(user33_over_birth_filled['birth'])
# Mengambil indeks data yang sama di kedua DataFrames
indexes = user33_over_birth_filled.index.intersection(user_SC33.index)

# Mengganti nilai pada copy_user93 dengan nilai yang sudah diperbaiki dari user93_over_birth_filled
user_SC33.loc[indexes, 'birth'] = user33_over_birth_filled.loc[indexes, 'birth']

In [ ]:
user_SC33 = user_SC33.drop('year', axis=1)

In [ ]:
user_SC33.shape

### SC61

In [ ]:
user_SC61 = user_new[user_new['id'].str.contains('SC61')]

In [ ]:
# Check for negative values in 'money_spent'
minus_ms = user_SC61[user_SC61['money_spent'] < 0]

# Check for negative values in 'refund'
minus_refund = user_SC61[user_SC61['refund'] < 0]

# Check for negative values in 'wallet_balance'
minus_wb = user_SC61[user_SC61['wallet_balance'] < 0]

# Print the number of rows with negative values
print(f"Number of rows with negative money_spent: {len(minus_ms)}")
print(f"Number of rows with negative refund: {len(minus_refund)}")
print(f"Number of rows with negative wallet_balance: {len(minus_wb)}")

In [ ]:
# Drop rows with negative refund
user_SC61 = user_SC61.drop(user_SC61[user_SC61['refund'] < 0].index)

# Drop rows with negative wallet balance
user_SC61 = user_SC61.drop(user_SC61[user_SC61['wallet_balance'] < 0].index)

In [ ]:
user61_over_birth = user_SC61[user_SC61['birth'] > '2006-12-31']

In [ ]:
# Membuat kolom baru 'year' yang berisi tahun dari kolom 'birth'
user_SC61['year'] = user_SC61['birth'].dt.year # Kolom year jika tidak diperlukan nantinya dapat dihapus

# Menghitung value counts untuk setiap tahun
value_counts_per_year = user_SC61['year'].value_counts().sort_index()

print(value_counts_per_year)

In [ ]:
# Membagi data menjadi 5 bagian
first_part_61 = user61_over_birth.iloc[:310]
second_part_61 = user61_over_birth.iloc[310:620]
third_part_61 = user61_over_birth.iloc[620:930]
fourth_part_61 = user61_over_birth.iloc[930:1240]
fifth_part_61 = user61_over_birth.iloc[1240:]

first_part_61['birth'] = '2002-03-03'
second_part_61['birth'] = '2003-04-04'
third_part_61['birth'] = '2004-05-05'
fourth_part_61['birth'] = '2005-06-06'
fifth_part_61['birth'] = '2006-07-07'

user61_over_birth_filled = pd.concat([first_part_61, second_part_61, third_part_61, fourth_part_61, fifth_part_61])

In [ ]:
# Mengubah tipe data menjadi datetime
user61_over_birth_filled['birth'] = pd.to_datetime(user61_over_birth_filled['birth'])
# Mengambil indeks data yang sama di kedua DataFrames
indexes = user61_over_birth_filled.index.intersection(user_SC61.index)

# Mengganti nilai pada copy_user93 dengan nilai yang sudah diperbaiki dari user93_over_birth_filled
user_SC61.loc[indexes, 'birth'] = user61_over_birth_filled.loc[indexes, 'birth']

In [ ]:
user_SC61 = user_SC61.drop('year', axis=1)

In [ ]:
user_SC61.shape

### SC62

In [ ]:
user_SC62 = user_new[user_new['id'].str.contains('SC62')]

In [ ]:
# Check for negative values in 'money_spent'
minus_ms = user_SC62[user_SC62['money_spent'] < 0]

# Check for negative values in 'refund'
minus_refund = user_SC62[user_SC62['refund'] < 0]

# Check for negative values in 'wallet_balance'
minus_wb = user_SC62[user_SC62['wallet_balance'] < 0]

# Print the number of rows with negative values
print(f"Number of rows with negative money_spent: {len(minus_ms)}")
print(f"Number of rows with negative refund: {len(minus_refund)}")
print(f"Number of rows with negative wallet_balance: {len(minus_wb)}")

In [ ]:
# Drop rows with negative refund
user_SC62 = user_SC62.drop(user_SC62[user_SC62['refund'] < 0].index)

# Drop rows with negative wallet balance
user_SC62 = user_SC62.drop(user_SC62[user_SC62['wallet_balance'] < 0].index)

In [ ]:
user62_over_birth = user_SC62[user_SC62['birth'] > '2006-12-31']

In [ ]:
# Membuat kolom baru 'year' yang berisi tahun dari kolom 'birth'
user_SC62['year'] = user_SC62['birth'].dt.year # Kolom year jika tidak diperlukan nantinya dapat dihapus

# Menghitung value counts untuk setiap tahun
value_counts_per_year = user_SC62['year'].value_counts().sort_index()

print(value_counts_per_year)

In [ ]:
first_part_62 = user62_over_birth.iloc[:100]
second_part_62 = user62_over_birth.iloc[100:200]
third_part_62 = user62_over_birth.iloc[200:300]
fourth_part_62 = user62_over_birth.iloc[300:400]
fifth_part_62 = user62_over_birth.iloc[400:500]
sixth_part_62 = user62_over_birth.iloc[500:]

first_part_62['birth'] = '2001-01-01'
second_part_62['birth'] = '2002-02-02'
third_part_62['birth'] = '2003-03-03'
fourth_part_62['birth'] = '2004-04-04'
fifth_part_62['birth'] = '2005-05-05'
sixth_part_62['birth'] = '2006-06-06'

user62_over_birth_filled = pd.concat([first_part_62, second_part_62, third_part_62, fourth_part_62, fifth_part_62, sixth_part_62])

In [ ]:
# Mengubah tipe data menjadi datetime
user62_over_birth_filled['birth'] = pd.to_datetime(user62_over_birth_filled['birth'])
# Mengambil indeks data yang sama di kedua DataFrames
indexes = user62_over_birth_filled.index.intersection(user_SC62.index)

# Mengganti nilai pada copy_user93 dengan nilai yang sudah diperbaiki dari user93_over_birth_filled
user_SC62.loc[indexes, 'birth'] = user62_over_birth_filled.loc[indexes, 'birth']

In [ ]:
user_SC62 = user_SC62.drop('year', axis=1)

In [ ]:
user_SC62.shape

## Data Digabungkan Kembali

In [ ]:
# concatenate the DataFrames
user_latest = pd.concat([user_SC33, user_SC61, user_SC62, user_SC93], ignore_index=True)

In [ ]:
user_latest.shape

## Feature Engineering

In [ ]:
# Buat kolom 'age'
user_latest['age'] = (pd.to_datetime('today') - user_latest['birth']).dt.days // 365

In [ ]:
# Buat kolom 'total_spent'
user_latest['total_spent'] = user_latest['money_spent'] - user_latest['refund']

In [ ]:
# Buat kolom 'saldo'
user_latest['balance'] = user_latest['wallet_balance'] + user_latest['refund']

In [ ]:
# Buat kolom 'harga_total'
user_latest['total_price'] = user_latest['gross_amount'] - user_latest['refund'] - user_latest['discounts']

In [ ]:
user_latest = pd.get_dummies(user_latest, columns=['gender'], dtype=int)

In [ ]:
user_latest = user_latest.rename(columns={'gender_laki-laki': 'male', 'gender_perempuan': 'female'})

In [ ]:
user_latest["rfm_recency"] = (pd.to_datetime('today') - user_latest["join_date"]).dt.days
user_latest["rfm_frequency"] = user_latest.groupby("id")["total_spent"].transform("count")
user_latest["rfm_monetary_value"] = user_latest["total_spent"].sum()
user_latest["tenure"] = (pd.to_datetime('today') - user_latest["join_date"]).dt.days
user_latest["refund_ratio"] = user_latest["refund"] / user_latest["total_spent"]

In [ ]:
user_latest.describe()

In [ ]:
user_latest = user_latest.drop(user_latest[user_latest['total_spent'] < 0].index)
user_latest = user_latest.drop(user_latest[user_latest['tenure'] < 0].index)
user_latest = user_latest.drop(user_latest[user_latest['total_price'] < 0].index)
user_latest = user_latest.drop('user_id', axis=1)

In [ ]:
user_latest = user_latest.drop(['join_date', 'birth'], axis=1)

In [ ]:
user_latest.dropna(inplace=True)

In [ ]:
user_latest.shape

## Iniliasasi

In [ ]:
# Install libraries (jika belum terinstall)
!pip install pycaret
from pycaret.clustering import *

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, MinMaxScaler, normalize
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

In [ ]:
# Normalize data dengan MinMax
MinMax = MinMaxScaler()

In [ ]:
cf = user_latest[['money_spent', 'balance', 'total_spent', 'gross_amount', 'rfm_recency', 'discounts', 'refund', 'refund_ratio']]
cf.replace([np.inf, -np.inf], np.nan, inplace=True)
cf_MinMax = MinMax.fit_transform(cf)

In [ ]:
cf_MinMax

In [ ]:
# Membuat dataframe dari hasil normalize untuk dilakukan invert terhadap
# beberapa kolom ['refund', 'discounts', 'rfm_recency']
cf_df = pd.DataFrame(cf_MinMax, columns=['money_spent', 'balance', 'total_spent',
                                         'gross_amount', 'rfm_recency', 'discounts',
                                         'refund', 'refund_ratio'])

In [ ]:
cf_df.head()

In [ ]:
# Invert fitur yang tidak linear
cf_df['refund'] = 1 - cf_df['refund']
cf_df['rfm_recency'] = 1 - cf_df['rfm_recency']
cf_df['discounts'] = 1 - cf_df['discounts']

In [ ]:
cf_df.head()

In [ ]:
cf_df.shape

In [ ]:
clustering_setup = setup(data=cf_df,
                         session_id=123)

## K-Means

In [ ]:
kmeans_model = create_model('kmeans', num_clusters=4)

In [ ]:
plot_model(kmeans_model, 'elbow')

In [ ]:
plot_model(kmeans_model, 'cluster')

## DBSCAN

In [ ]:
dbscan_model = create_model('dbscan', num_clusters=4)

NameError: name 'create_model' is not defined

In [ ]:
plot_model(dbscan_model, 'elbow')

In [ ]:
plot_model(dbscan_model, 'cluster')

## Assign Model

In [ ]:
kmeans_df = assign_model(kmeans_model)

In [ ]:
user_latest['Cluster'] = kmeans_df['Cluster']

In [ ]:
user_latest.head()

In [ ]:
# Membagi data menjadi k bagian sesuai dengan banyaknya cluster
cluster_0 = user_latest.loc[user_latest['Cluster'] == 'Cluster 0']
cluster_1 = user_latest.loc[user_latest['Cluster'] == 'Cluster 1']
cluster_2 = user_latest.loc[user_latest['Cluster'] == 'Cluster 2']
cluster_3 = user_latest.loc[user_latest['Cluster'] == 'Cluster 3']

In [ ]:
cluster_0.describe()

In [ ]:
cluster_1.describe()

In [ ]:
cluster_2.describe()

In [ ]:
cluster_3.describe()

In [ ]:
# Men-drop beberapa data yang tidak masuk ke cluster manapun
user_latest = user_latest.drop(user_latest.loc[user_latest['Cluster'].isnull()].index)

# SC93 = 107 data (semua data pada provinsi 93)
# SC62 = 449 data

In [ ]:
user_latest.shape

In [ ]:
user_latest.to_csv('user_new_per_cluster_final.csv', index=False)